# Error analysis — Day 9

Symptom → root cause → the experiment run → whether it worked. That shape is what
section 10 asks for, and it is the format carried over from the voice-agent work
because it is the one that survives contact with a reviewer.

Weighted toward **false interruptions**: at a 10% interruption ceiling a uniform
sample of failures would be almost all missed endpoints and would teach nothing.

Run first:

```
python scripts/error_analysis.py --checkpoint weights/E1-best.pt --mine-hard-negatives
```

In [1]:
import warnings, sys, json, collections
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src import SAMPLE_RATE
from src.dataset import WaveCache
from src.inference import TurnPredictor
from src.evaluation import evaluate_predictor, slice_report
from src.metrics import confusion_at, operating_points

CHECKPOINT = "../weights/E1-best.pt"
CACHE = "../data/cache/test"

# This notebook is Day 9 and depends on Day 5, so being run early is expected
# rather than an error. Fall back to any checkpoint that does exist, and say
# plainly what is missing instead of throwing a traceback.
READY = Path(CHECKPOINT).exists()
if not READY:
    found = sorted(Path("../weights").glob("*-best.pt")) if Path("../weights").exists() else []
    if found:
        CHECKPOINT, READY = str(found[-1]), True
        print(f"E1-best.pt not present — using {CHECKPOINT}")
if not READY:
    print("No trained checkpoint in ../weights/. Run:")
    print("  python scripts/prepare_data.py --split train --max-rows 40000")
    print("  python -m training.train --config configs/e1_frozen_linear.yaml")
    print()
    print("Every cell below will skip cleanly until then.")
else:
    print(f"checkpoint: {CHECKPOINT}")

No trained checkpoint in ../weights/. Run:
  python scripts/prepare_data.py --split train --max-rows 40000
  python -m training.train --config configs/e1_frozen_linear.yaml

Every cell below will skip cleanly until then.


In [2]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    pred = TurnPredictor(CHECKPOINT, backend="torch")
    cache = WaveCache(CACHE)
    idx = np.arange(len(cache))
    print(pred.info())
    print(cache.summary())

    ev, probs = evaluate_predictor(pred, cache, idx, "E1", threshold=pred.threshold)
    thr = ev.threshold
    y = np.asarray([cache.label(int(i)) for i in idx])
    conf = confusion_at(y, probs, thr)
    print()
    print(conf)
    print()
    print(conf.matrix_str())

skipped — no trained checkpoint yet (see cell 1)


## 1. Where does the score distribution actually sit?

In [3]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for l, nm, c in [(0, "not endpoint", "#A93A28"), (1, "endpoint", "#2E7358")]:
        ax[0].hist(probs[y == l], bins=50, alpha=0.6, label=nm, color=c)
    ax[0].axvline(thr, color="#14172A", ls="--", label=f"threshold {thr:.3f}")
    ax[0].set_xlabel("P(turn ended)")
    ax[0].legend()
    ax[0].set_title("score distribution — the overlap is the error budget")

    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y, probs)
    ax[1].plot(fpr, tpr, color="#4A45C9")
    ax[1].plot([0, 1], [0, 1], ls=":", color="#5B6280")
    ax[1].axvline(0.10, color="#9C6410", ls="--", label="10% interruption ceiling")
    ax[1].set_xlabel("false interruption rate (FPR)")
    ax[1].set_ylabel("recall (TPR)")
    ax[1].legend()
    ax[1].set_title(f"ROC — AUC {ev.roc_auc:.4f}")
    plt.tight_layout()
    plt.show()

    print(json.dumps(operating_points(y, probs), indent=2)[:1400])

skipped — no trained checkpoint yet (see cell 1)


## 2. The failures, most-confident first

A confidently wrong prediction is far more informative than a borderline one:
borderline errors are the threshold's fault, confident errors are the model's.

In [4]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    pred_pos = probs >= thr
    fi = np.flatnonzero(pred_pos & (y == 0))        # false interruptions
    me = np.flatnonzero((~pred_pos) & (y == 1))    # missed endpoints
    fi = fi[np.argsort(-probs[fi])]
    me = me[np.argsort(probs[me])]
    print(f"{fi.size} false interruptions, {me.size} missed endpoints")


    def show(sel, title, k=6):
        print()
        print(f"=== {title} ===")
        for j in sel[:k]:
            m = cache.meta[int(idx[j])]
            w = cache.wave(int(idx[j]))
            print(f"p={probs[j]:.3f} thr={thr:.3f} | {m['language']}/{m['dataset']} "
                  f"{w.size / SAMPLE_RATE:.2f}s midfill={m.get('midfiller')} "
                  f"endfill={m.get('endfiller')} synth={m.get('synthetic')}")
            display(Audio(w, rate=SAMPLE_RATE))


    show(fi, "FALSE INTERRUPTIONS — the model cut the speaker off")

skipped — no trained checkpoint yet (see cell 1)


In [5]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    show(me, "MISSED ENDPOINTS — the model sat through a finished turn")

skipped — no trained checkpoint yet (see cell 1)


## 3. Categorise the modes, using the corpus's own annotations

The `midfiller` / `endfiller` flags turn "the model struggles with hesitation"
from an impression into a number.

In [6]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    def rate(mask_rows, want_label):
        m = mask_rows & (y == want_label)
        if m.sum() < 10:
            return None, int(m.sum())
        if want_label == 0:
            return float((pred_pos & m).sum() / m.sum()), int(m.sum())
        return float(((~pred_pos) & m).sum() / m.sum()), int(m.sum())


    groups = {
        "endfiller=True": np.asarray([cache.meta[int(i)].get("endfiller") is True for i in idx]),
        "midfiller=True": np.asarray([cache.meta[int(i)].get("midfiller") is True for i in idx]),
        "no filler (annot.)": np.asarray([
            cache.meta[int(i)].get("midfiller") is False
            and cache.meta[int(i)].get("endfiller") is False for i in idx]),
        "synthetic=True": np.asarray([cache.meta[int(i)].get("synthetic") is True for i in idx]),
        "synthetic=False": np.asarray([cache.meta[int(i)].get("synthetic") is False for i in idx]),
    }
    print(f"{'group':<22s} {'FI rate':>9s} {'n_neg':>7s}   {'missed':>8s} {'n_pos':>7s}")
    for name, m in groups.items():
        fir, nneg = rate(m, 0)
        mer, npos = rate(m, 1)
        fi_s = "      n/a" if fir is None else f"{fir:9.4f}"
        me_s = "     n/a" if mer is None else f"{mer:8.4f}"
        print(f"{name:<22s} {fi_s} {nneg:>7d}   {me_s} {npos:>7d}")
    print()
    print("A high FI rate on endfiller=True is the model cutting off trailing")
    print("fillers — the single most user-visible failure this task has.")

skipped — no trained checkpoint yet (see cell 1)


## 4. Per-language and per-source breakdown

In [7]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    import pandas as pd

    rows = slice_report(cache, idx, probs, thr)
    df = pd.DataFrame(rows)
    display(df[["slice", "n", "positive_rate", "f1", "recall", "false_interrupt", "missed"]])

skipped — no trained checkpoint yet (see cell 1)


In [8]:
if not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    lang_rows = [r for r in rows if r["slice"].startswith("lang=")]
    if lang_rows:
        lang_rows.sort(key=lambda r: r["false_interrupt"])
        names = [r["slice"].replace("lang=", "") for r in lang_rows]
        fig, ax = plt.subplots(figsize=(10, 3.4))
        ax.bar(names, [r["false_interrupt"] for r in lang_rows], color="#A93A28", alpha=0.8)
        ax.axhline(0.10, ls="--", color="#14172A", label="10% ceiling")
        ax.set_ylabel("false interruption rate")
        ax.set_title("false interruptions by language — a tall bar is a language being cut off")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("no language slice met the minimum sample count")

skipped — no trained checkpoint yet (see cell 1)


## 5. Write the findings up

One row per mode. The last column is what makes this section rigour rather than
description — and a fix that **did not work** stays in the table.

| symptom | root cause | experiment run | worked? |
|---|---|---|---|
| _(fill from section 3)_ | | | |
| | | | |
| | | | |

## 6. Hard-negative mining, then the retrain

`scripts/error_analysis.py --mine-hard-negatives` writes the indices. Oversample
them in the next run and report the outcome **either way**.

In [9]:
p = Path("../artifacts/runs/error_analysis/hard_negative_indices.npy")
if not p.exists():
    print("Run: python scripts/error_analysis.py --checkpoint weights/E1-best.pt "
          "--mine-hard-negatives")
elif not READY:
    print("skipped — no trained checkpoint yet (see cell 1)")
else:
    hard = np.load(p)
    print(f"{hard.size} hard negatives mined")
    print("by source:", dict(collections.Counter(
        cache.meta[int(i)]["dataset"] for i in hard).most_common(8)))
    print("by language:", dict(collections.Counter(
        cache.meta[int(i)]["language"] for i in hard).most_common(8)))
    print()
    print("A concentration in one source or language tells you what the next")
    print("training run is actually short of.")

Run: python scripts/error_analysis.py --checkpoint weights/E1-best.pt --mine-hard-negatives


### The retrain result goes here

After training with these oversampled, record the comparison. Report it even if
it is worse — a documented failed fix reads as rigour; a silently dropped one
reads as cherry-picking.

| run | F1 | false interruption | missed | verdict |
|---|---|---|---|---|
| E5 (before) | | | | |
| E6 (+hard negatives) | | | | |